# `dataeval` Shift Tutorial

A guide to running the `dataeval` shift tools via the Reference Implementation (RI).

> **NOTE:** The `dataeval` package can be used in the RI framework for both image classification (IC) and object detection (OD) tasks. This tutorial will only cover the OD scenario.

## What is `dataeval`?

The `dataeval` package analyzes datasets and models to give users the ability to train and test performant, unbiased, and reliable AI models and monitor data for impactful shifts to deployed models.

The tools demonstrated in this tutorial are a subset of the larger `dataeval` framework. They are specifically focused on identifying statistical differences between operational data and training data. A common cause of model degradation in an operational setting is a significant deviation between the operational and training data.

At a high-level, identifying statistical differences between datasets involves:

* Using an AI model to generate mathematical representations of the different datasets
* Performing a range of statistical tests to determine whether there are differences between these mathematical representations

The `dataeval` shift tools are generally applied to an entire dataset. Their computational demands are moderate, and are run on both CPU and GPU. (The AI models are run on GPU, but they can also be run on CPU if a GPU is not available.) 

## Overview and Background

This section will outline the aspects most relevant to applying the `dataeval` shift tools to Object Detection problems similar to the RI's use case.

For more in-depth reading on the tool, visit the [`dataeval` documentation](https://jatic.pages.jatic.net/aria/dataeval/index.html)

## Using an AI model to generate a mathematical representation of a dataset

`dataeval` supports the use of AI for generating mathematical representations of a dataset, and provides some basic AI models for this task. Power-users can also bring their own AI models if required.

As this is a reasonably technical topic, it is recommended to refer to the `dataeval` documentation for precise details.

## Detecting Dataset Drift

Drift refers to the phenomenon where the statistical properties of data change over time, leading to discrepancies between the data a model was trained on and the data it encounters during deployment. This can significantly degrade the performance of machine learning models, as the assumptions made during training may no longer hold in real-world scenarios. 

`dataeval` provide three different statistical tests for detecting dataset drift:

- **Cramér-von Mises**

- **Kolmogorov-Smirnov**

- **Maximum Mean Discrepancy**

![](../assets/dataeval_shift_example_drift.png)

## Detecting Out-of-Distribution Data

Out-of-distribution (OOD) detectors identify operational data that is different from the data used to train a particular model. This is a reasonably advanced topic that uses a specially designed set of AI models to de-construct and then re-construct an operational dataset. The re-constructed dataset is then compared against the original dataset. If there are large differences, then this is evidence that suggests that the operational data is substantially different to the training dataset.

![](../assets/dataeval_shift_example_ood.png)

## Running the `dataeval` shift detection algorithms inside the JATIC RI

The following section uses the JATIC Reference Implementation API to run the `dataeval` shift test stage for Object Detection.

First, we create the the necessary MAITE-wrapped datasets. We use the `CocoDetectionDataset` wrapper.  The data is found in our test directory.

In [ ]:
from pathlib import Path
from jatic_ri.object_detection.datasets import CocoDetectionDataset

BASE_DIR = Path.cwd().parents[1]
dataset_root_path = BASE_DIR / "tests/testing_utilities/example_data/coco_resized_val2017"
dataset_ann_file_path = BASE_DIR / "tests/testing_utilities/example_data/coco_resized_val2017/instances_val2017_resized_6.json"

print("Loading training COCO dataset...")
dataset_train = CocoDetectionDataset(root=dataset_root_path, ann_file=dataset_ann_file_path, dataset_id="coco-train")
print(f"Dataset loaded with {len(dataset_train)} images")

In [ ]:
from pathlib import Path
from jatic_ri.object_detection.datasets import CocoDetectionDataset

BASE_DIR = Path.cwd().parents[1]
dataset_root_path = BASE_DIR / "tests/testing_utilities/example_data/coco_dataset"
dataset_ann_file_path = BASE_DIR / "tests/testing_utilities/example_data/coco_dataset/ann_file.json"

print("Loading operational COCO dataset...")
dataset_operational = CocoDetectionDataset(root=dataset_root_path, ann_file=dataset_ann_file_path, dataset_id="coco-op")
print(f"Dataset loaded with {len(dataset_operational)} images")

Next, we initialize an `DatasetShiftTestStage` object, load the dataset wrapped above, and execute the test stage.

In [ ]:
from jatic_ri.object_detection.test_stages import DatasetShiftTestStage

test_stage = DatasetShiftTestStage()
output = test_stage.run(use_stage_cache=False, datasets=[dataset_train, dataset_operational])

## Slide Deck

Once the test stage has completed, the code below uses the `gradient` package to create HTML and PPTX formatted reports of the results of the `dataeval` shift test stage.

In [ ]:
import os
from gradient.templates_and_layouts.create_deck import create_deck

output_dir = Path("dataeval_shift_example_output")
os.makedirs(output_dir, exist_ok=True)

create_deck(output.collect_report_consumables(threshold=0), output_dir, deck_name='Dataeval_Shift_Example_Report')
print(f"PowerPoint saved in {output_dir}.")